## Plot For Bool Q Dataset

In [ ]:
import re
from typing import Dict
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
from pathlib import Path

from multi_llm_debate.analysis.correct_rate_by_round import (
    calculate_correct_rate_by_round,
)
from multi_llm_debate.analysis.calculate_task_accuracy import analyze_task_accuracy
from multi_llm_debate.run.bool_q.utils import extract_bool_answer

# ==========================
# 1. CONFIGURATION
# ==========================
DATA_PATH = Path("../output/bool_q/processed_data.csv")
bool_q_path = Path("../datasets/bool_q")
MODEL_DIR_PATH = Path("../data/bool_q")

# We only want directories whose parentheses-based digit sum == 6
TARGET_MODEL_COUNT = 11

# Maximum round number for your correct_rate_by_round function
MAX_ROUND_NUMBER = 10


# ==========================
# 2. HELPER FUNCTIONS
# ==========================
def get_total_model_count(dir_name: str) -> int:
    """Parses folder name to sum up the numeric values found in parentheses.
    
    Args:
        dir_name: Model directory name, e.g., 'llama3(3)+mistral(3)'
        
    Returns:
        Sum of all numbers found in parentheses
    """
    matches = re.findall(r"\((\d+)\)", dir_name)
    return sum(int(m) for m in matches)


def create_plot(
    accuracies_by_round: Dict[float, Dict[str, np.ndarray]], 
    model_name: str
) -> None:
    """Plots lines for each accuracy value and metric type.
    
    Args:
        accuracies_by_round: Dictionary mapping accuracy values to a dict of
            metrics with their corresponding values.
        model_name: Name of the model for the plot title.
    """
    # Sort the dictionary items by the accuracy value (the dictionary key)
    sorted_items = sorted(accuracies_by_round.items(), key=lambda x: x[0])
    
    # Use a color map for different accuracy values
    accuracy_colors = plt.cm.get_cmap('tab20', len(sorted_items))
    
    # Fixed colors for the different metrics
    metric_color_map = {
        'absolute': '#1f77b4',  # Blue
        'majority': '#ff7f0e',  # Orange
        'majority_vote': '#2ca02c',  # Green - alternative name if needed
    }

    plt.figure(figsize=(10, 6))
    
    # Define line styles for different metrics
    line_styles = {
        'absolute': '-',       # solid line
        'majority': '--',      # dashed line
        'majority_vote': '--'  # dashed line (alternative name)
    }
    
    legend_handles = []

    # Plot a line for each unique accuracy value and metric
    for idx, (accuracy, metrics_dict) in enumerate(sorted_items):
        for metric_name, values in metrics_dict.items():
            # Skip empty arrays or None values
            if values is None or len(values) == 0:
                continue
                
            # Create x-axis values matching the length of values
            rounds = np.arange(len(values))
            
            # Only plot if both rounds and values have the same length
            if len(rounds) > 0 and len(rounds) == len(values):
                # Get color for this specific metric
                if metric_name in metric_color_map:
                    color = metric_color_map[metric_name]
                else:
                    # Fallback to accuracy-based color if metric not in map
                    color = accuracy_colors(idx)
                
                line, = plt.plot(
                    rounds,
                    values,
                    color=color,
                    linestyle=line_styles.get(metric_name, '-'),
                    linewidth=2,
                    label=f"Acc={accuracy:.2f} ({metric_name})"
                )
                legend_handles.append(line)
                

    # Title and labels
    plt.title(f'Accuracy by Round: {model_name}', pad=15)
    plt.xlabel('Round Number')
    plt.ylabel('Correct Rate')
    plt.grid(True, linestyle='--', alpha=0.7)
    
    if legend_handles:
        plt.legend(handles=legend_handles)

    # Set y-axis limits and ticks
    plt.ylim(0, 1)
    plt.yticks(np.arange(0, 1.1, 0.1))
    plt.xticks(range(min(11, MAX_ROUND_NUMBER + 1)))

    plt.tight_layout()
    plt.show()


def process_model(model_dir: Path) -> None:
    """Process model data and create visualizations.
    
    Args:
        model_dir: Path to the model directory containing debate data.
    
    This function:
    1) Analyzes accuracy
    2) Calculates correct rates for each unique accuracy value
    3) Prints the percentage of tasks for each accuracy value
    4) Plots the results for both absolute and majority vote metrics
    """
    model_name = model_dir.name
    print(f"\nProcessing model: {model_name}")

    # 1) Analyze accuracy
    result_df = analyze_task_accuracy(
        model_dir=model_dir,
        dataframe=df,
        extract_fn=extract_bool_answer,
    )

    # 2) Get all unique accuracy values from the result dataframe
    unique_accuracies = result_df['accuracy'].unique()

    # 3) Create a dictionary to store the metrics by round for each accuracy
    accuracies_by_round = {
        accuracy: {'absolute': None, 'majority_vote': None} 
        for accuracy in unique_accuracies
    }

    length = len(result_df)
    # 4) For each unique accuracy value, calculate metrics by round
    for accuracy in unique_accuracies:
        if accuracy < 0:
            continue
            
        # Filter tasks by accuracy
        filtered_df = result_df[result_df['accuracy'] == accuracy]
        
        # Calculate and print the percentage of tasks with this accuracy
        accuracy_percentage = (len(filtered_df) / length) * 100
        print(f"Accuracy = {accuracy:.2f}: {accuracy_percentage:.2f}% of total tasks")

        try:
            # Calculate correct rates for this accuracy
            cr_filtered_df = calculate_correct_rate_by_round(
                filtered_df, 
                model_dir, 
                max_round_number=MAX_ROUND_NUMBER, 
                extract_func=extract_bool_answer
            )
            
            # Check if we have results for the metrics
            absolute_rows = cr_filtered_df[cr_filtered_df['metric'] == 'absolute']
            if not absolute_rows.empty:
                accuracies_by_round[accuracy]['absolute'] = absolute_rows.iloc[0, 2:].values
                
            majority_rows = cr_filtered_df[cr_filtered_df['metric'] == 'majority']
            if not majority_rows.empty:
                accuracies_by_round[accuracy]['majority'] = majority_rows.iloc[0, 2:].values
                
        except Exception as e:
            print(f"Error processing accuracy {accuracy}: {e}")
            continue

    # 5) Create the plot
    create_plot(accuracies_by_round, model_name)


# ==========================
# 3. MAIN SCRIPT
# ==========================
if __name__ == "__main__":
    from multi_llm_debate.run.bool_q.utils import process_bool_q_df
    from multi_llm_debate.utils.download_dataset import load_save_dataset_df
    import os
    
    # Load the main data
    if not DATA_PATH.exists():
        os.makedirs(DATA_PATH.parent, exist_ok=True)
        dataset = load_save_dataset_df(
            dataset_name="google/boolq",
            dataset_path=bool_q_path,
            force_download=False,
        )
        df = process_bool_q_df(dataset)
        df.to_csv(DATA_PATH, index=False)
    else:
        df = pd.read_csv(DATA_PATH)
        
    print("DF columns:", df.columns)
    
    # Get all model directories
    all_model_dirs = list(MODEL_DIR_PATH.glob('*'))

    # Filter directories by total model count == TARGET_MODEL_COUNT
    filtered_model_dirs = [
        d for d in all_model_dirs
        if get_total_model_count(d.name) == TARGET_MODEL_COUNT
    ]

    print(f"Filtered directories (sum of parentheses == {TARGET_MODEL_COUNT}):")
    for d in filtered_model_dirs:
        print("  -", d.name)

    # Process each filtered directory
    for model_dir in filtered_model_dirs:
        process_model(model_dir)

In [ ]:
from pathlib import Path
import matplotlib.pyplot as plt
from multi_llm_debate.analysis.calculate_correct_rate_distribution import (
    calculate_correct_rate_distribution_for_round_n,
)
from multi_llm_debate.run.bool_q.utils import extract_bool_answer
from multi_llm_debate.analysis.utils import compare_bool
import pandas as pd
import seaborn as sns
import logging

from typing import Dict, List, Tuple

# Set up logging
logging.basicConfig(
    level=logging.INFO,
    format="%(asctime)s - %(name)s - %(levelname)s - %(message)s",
)
logger = logging.getLogger(__name__)

def process_distribution_data(
    result_df: pd.DataFrame,
    round_number: int,
) -> Dict[str, float]:
    """Process distribution data to get percentages for each bin.
    
    Args:
        result_df: DataFrame with distribution data.
        round_number: The round number being processed.
        
    Returns:
        Dictionary mapping bin labels to percentages.
    """
    # Get bin columns (those that are digits)
    bin_columns = [col for col in result_df.columns if col.isdigit()]
    bin_columns.sort(key=int)  # Sort numerically
    
    if not bin_columns or result_df.empty:
        logger.warning(f"No bins found for round {round_number}")
        return {}
    
    # Count tasks and calculate percentages
    task_count = len(result_df)
    bin_sums = result_df[bin_columns].sum()
    bin_percentages = (bin_sums / task_count * 100).to_dict()
    
    return bin_percentages

def plot_round_distribution(
    bin_percentages: Dict[str, float],
    round_number: int,
    output_dir: Path,
    show_plot: bool = False,
) -> None:
    """Create and save a plot of the correct rate distribution for a round.
    
    Args:
        bin_percentages: Dictionary mapping bin labels to percentage values.
        round_number: The round number being visualized.
        output_dir: Directory where the plot should be saved.
        show_plot: Whether to display the plot interactively.
    """
    if not bin_percentages:
        logger.warning(f"No data to plot for round {round_number}")
        return
    
    plt.figure(figsize=(10, 6))
    
    # Sort bins numerically
    bins = [int(b) for b in sorted(bin_percentages.keys(), key=int)]
    values = [bin_percentages[str(b)] for b in bins]
    
    # Create bar chart
    bars = plt.bar(bins, values)
    
    # Add value labels on top of bars
    for bar in bars:
        height = bar.get_height()
        plt.text(
            bar.get_x() + bar.get_width()/2.,
            height + 1,
            f'{height:.1f}%',
            ha='center',
            fontsize=9,
        )
    
    # Set chart attributes
    plt.title(f'Round {round_number}: Distribution of Correct Agents', 
             fontsize=14)
    plt.xlabel('Number of Correct Agents', fontsize=12)
    plt.ylabel('Percentage of Tasks (%)', fontsize=12)
    plt.grid(axis='y', alpha=0.3)
    plt.ylim(0, max(values) * 1.2)  # Add some headroom for labels
    
    # Save the plot
    output_path = output_dir / f"round_{round_number}_distribution.png"
    plt.tight_layout()
    plt.savefig(output_path, dpi=300)
    
    if show_plot:
        plt.show()
    plt.close()
    
    logger.info(f"Saved plot for round {round_number} to {output_path}")

def create_heatmap(
    all_distributions: List[Tuple[int, Dict[str, float]]],
    output_dir: Path,
    show_plot: bool = False,
) -> None:
    """Create a heatmap showing the evolution of distributions across rounds.
    
    Args:
        all_distributions: List of (round_number, bin_percentages) tuples.
        output_dir: Directory where the plot should be saved.
        show_plot: Whether to display the plot interactively.
    """
    if not all_distributions:
        logger.warning("No data to create heatmap")
        return
    
    # Create a DataFrame from the collected data
    data = []
    for round_num, bin_percentages in all_distributions:
        for bin_label, percentage in bin_percentages.items():
            data.append({
                'Round': round_num,
                'Correct Agents': int(bin_label),
                'Percentage': percentage
            })
    
    df = pd.DataFrame(data)
    
    # Create pivot table for heatmap
    pivot_df = df.pivot(
        index='Round', 
        columns='Correct Agents', 
        values='Percentage'
    ).fillna(0)
    
    # Create heatmap plot
    plt.figure(figsize=(12, 8))
    ax = sns.heatmap(
        pivot_df,
        annot=True,
        fmt=".1f",
        cmap="YlGnBu",
        linewidths=0.5,
        cbar_kws={'label': 'Percentage of Tasks (%)'}
    )
    
    plt.title('Evolution of Correct Agent Distribution Across Rounds', 
             fontsize=16)
    plt.tight_layout()
    
    # Save the heatmap
    output_path = output_dir / "correct_agent_distribution_heatmap.png"
    plt.savefig(output_path, dpi=300)
    
    if show_plot:
        plt.show()
    plt.close()
    
    logger.info(f"Saved heatmap to {output_path}")

def main(
    data_path: Path,
    model_dir: Path,
    output_dir: Path,
    max_rounds: int = 6,
    show_plots: bool = False,
) -> None:
    """Run the visualization process for correct rate distributions.
    
    Args:
        data_path: Path to the CSV file with task data.
        model_dir: Directory containing model output data.
        output_dir: Directory where output plots should be saved.
        max_rounds: Maximum number of rounds to process.
        show_plots: Whether to display plots interactively.
    """
    # Create output directory if it doesn't exist
    output_dir.mkdir(parents=True, exist_ok=True)
    
    try:
        # Load answer data (contains correct labels)
        df_answers = pd.read_csv(data_path)
        # Ensure ID column is numeric
        df_answers["id"] = pd.to_numeric(df_answers["id"], errors="coerce")
        df_answers.dropna(subset=["id"], inplace=True)
        df_answers["id"] = df_answers["id"].astype(int)
        logger.info(f"Loaded answer data from {data_path}")
        
        # Load debate data
        debates_path = model_dir / "debate_rounds.csv"
        df_debates = pd.read_csv(debates_path)
        # Ensure numeric columns
        df_debates["task_id"] = pd.to_numeric(df_debates["task_id"], errors="coerce")
        df_debates["round_number"] = pd.to_numeric(
            df_debates["round_number"], errors="coerce"
        )
        df_debates.dropna(subset=["task_id", "round_number"], inplace=True)
        df_debates["task_id"] = df_debates["task_id"].astype(int)
        df_debates["round_number"] = df_debates["round_number"].astype(int)
        logger.info(f"Loaded debate data from {debates_path}")
        
    except Exception as e:
        logger.error(f"Error loading data: {e}")
        return
    
    all_distributions = []
    
    # Process each round
    for round_number in range(max_rounds):
        logger.info(f"Processing round {round_number}...")
        
        # Calculate distribution with the required parameters
        result_df = calculate_correct_rate_distribution_for_round_n(
            df_answers=df_answers,
            df_debates=df_debates,
            round_number=round_number,
            extract_func=extract_bool_answer,
            compare_func=compare_bool,
        )
        
        # Process and store the distribution data
        bin_percentages = process_distribution_data(result_df, round_number)
        
        if bin_percentages:
            all_distributions.append((round_number, bin_percentages))
            
            # Create individual round plot
            plot_round_distribution(
                bin_percentages,
                round_number,
                output_dir,
                show_plot=show_plots,
            )
    
    # Create summary heatmap visualization
    create_heatmap(all_distributions, output_dir, show_plot=show_plots)
    
    logger.info("Visualization complete!")


if __name__ == "__main__":
    # Define paths
    DATA_PATH = Path("../output/bool_q/processed_data.csv")
    MODEL_DIR_PATH = Path("../data/bool_q/llama3(11)")
    OUTPUT_DIR = Path("../output/visualizations")

    # Run the main function
    main(
        data_path=DATA_PATH,
        model_dir=MODEL_DIR_PATH,
        output_dir=OUTPUT_DIR,
        max_rounds=6,
        show_plots=True,  # Set to True to display plots interactively
    )

## Plot For JudgeBench

### Round Number Distribution

In [ ]:
from pathlib import Path
import matplotlib.pyplot as plt
import os
from collections import Counter
from typing import Dict

def count_files_per_directory(base_dir_path: str) -> Dict[int, int]:
    """Counts how many directories contain each number of files.
    
    Args:
        base_dir_path: Path to the base directory containing the data.
        
    Returns:
        A dictionary mapping file counts to the number of directories with that count.
    """
    base_path = Path(base_dir_path)
    file_counts = []
    
    # Check if the directory exists
    if not base_path.exists():
        print(f"Directory not found: {base_dir_path}")
        return {}
        
    # Walk through all directories and count their files
    for root, dirs, files in os.walk(base_path):
        file_counts.append(len(files))
    
    # Count how many directories have each file count
    distribution = Counter(file_counts)
    
    return dict(sorted(distribution.items()))

def plot_file_count_distribution(distribution: Dict[int, int]) -> None:
    """Creates and displays a plot of file count distribution across directories.
    
    Args:
        distribution: Dictionary mapping file counts to number of directories.
    """
    if not distribution:
        print("No directory data found to plot.")
        return
    
    plt.figure(figsize=(12, 6))
    
    # Prepare data
    file_counts = list(distribution.keys())
    dir_counts = list(distribution.values())
    
    # Create bar chart
    bars = plt.bar(file_counts, dir_counts, color='salmon', edgecolor='darkred')
    
    # Add value labels on top of bars
    for bar in bars:
        height = bar.get_height()
        plt.text(
            bar.get_x() + bar.get_width()/2.,
            height + 0.1,
            f'{int(height)}',
            ha='center',
            fontsize=9,
        )
    
    # Set chart attributes
    plt.title('Distribution of File Counts Across Directories', fontsize=14)
    plt.xlabel('Number of Files in Directory', fontsize=12)
    plt.ylabel('Number of Directories', fontsize=12)
    plt.grid(axis='y', alpha=0.3)
    
    # Adjust x-axis to show all integer values
    plt.xticks(range(min(file_counts), max(file_counts)+1))
    
    plt.tight_layout()
    plt.show()

def main(model_dir_path: str) -> None:
    """Main function to analyze and visualize file count distribution.
    
    Args:
        model_dir_path: Path to the model directory containing data.
    """
    print(f"Analyzing files in: {model_dir_path}")
    
    # Get file count distribution
    distribution = count_files_per_directory(model_dir_path)
    
    # Display summary
    if distribution:
        print("\nFile count distribution across directories:")
        print(f"{'Files':<10} {'Directories':>12}")
        print("-" * 22)
        for count, num_dirs in sorted(distribution.items()):
            print(f"{count:<10} {num_dirs:>12}")
        
        total_dirs = sum(distribution.values())
        print(f"\nTotal directories analyzed: {total_dirs}")
        print(f"Average files per directory: {sum(k*v for k,v in distribution.items())/total_dirs:.2f}")
    else:
        print("No directory data found.")
    
    # Create visualization
    plot_file_count_distribution(distribution)

if __name__ == "__main__":
    model_dir = "../data/judge_bench/llama3(11)"
    main(model_dir)
    model_dir = "../data/judge_bench/gemma2:2b(11)"
    main(model_dir)

### Accuracy by Round

In [1]:
import os
import json
import logging
import math
from pathlib import Path
from typing import Dict, List, Tuple, Optional

import pandas as pd
import matplotlib.pyplot as plt
import seaborn as sns

# -------------------------------------------------------
# External imports from your multi_llm_debate package
# -------------------------------------------------------
from multi_llm_debate.analysis.calculate_correct_rate_distribution import (
    calculate_correct_rate_distribution_for_round_n,
)
from multi_llm_debate.run.judge_bench.utils import (
    extract_caption_a_b_answer,
    compare_judge_bench_responses,
    load_judge_bench_dataset,
)

# -------------------------------------------------------
# Configure Logging
# -------------------------------------------------------
logging.basicConfig(
    level=logging.INFO,
    format="%(asctime)s - %(name)s - %(levelname)s - %(message)s",
)
logger = logging.getLogger(__name__)

# -------------------------------------------------------
# Load Debate Data
# -------------------------------------------------------
def load_debate_data(model_dir: Path, write_csv: bool = True) -> Optional[pd.DataFrame]:
    """Loads debate data from model directory.

    This function attempts to find and load debate data, first looking for
    a debate_rounds.csv file, then searching for debate data in directories
    organized by task IDs and round files. If the CSV file doesn't exist and
    write_csv is True, it will create the file for future use.

    Args:
        model_dir: Directory containing model output data.
        write_csv: If True, writes a consolidated debate_rounds.csv file when
                   one doesn't already exist. Defaults to True.

    Returns:
        DataFrame with debate data or None if data cannot be found/loaded.
    """
    if not model_dir.is_dir():
        logger.error(f"Model directory not found: {model_dir}")
        return None

    debates_path = model_dir / "debate_rounds.csv"
    if debates_path.exists():
        logger.info(f"Found debate_rounds.csv at {debates_path}")
        return pd.read_csv(debates_path)

    # If not found, try to construct data from directories
    logger.info("No debate_rounds.csv found. Attempting to load from directories.")

    all_data = []

    # Check for task directories (e.g., "task_1", "task_2", etc.)
    for task_dir in model_dir.glob("*"):
        if task_dir.is_dir():
            try:
                # Process all debate_round_*.json files in this task directory
                for debate_file in task_dir.glob("debate_round_*.json"):
                    try:
                        # Example format: debate_round_3.json -> round_num = 3
                        round_num = int(debate_file.stem.split("_")[2])
                        logger.info(f"Processing file: {debate_file.name} (Round {round_num})")

                        # Open and read the JSON file
                        with open(debate_file, "r") as f:
                            task_data = json.load(f)

                        # Handle JSON array of agent responses
                        if isinstance(task_data, list):
                            for agent_data in task_data:
                                if "agent_id" in agent_data and "response" in agent_data:
                                    # Attempt to extract a numeric task_id from folder
                                    folder_name = task_dir.name
                                    if folder_name.startswith("task_"):
                                        numeric_id_str = folder_name.replace("task_", "")
                                        try:
                                            numeric_id = int(numeric_id_str)
                                        except ValueError:
                                            numeric_id = None
                                    else:
                                        # Fallback to string if no "task_" prefix
                                        numeric_id = folder_name

                                    all_data.append(
                                        {
                                            "task_id": numeric_id,
                                            "round_number": round_num,
                                            "agent_id": agent_data["agent_id"],
                                            "model": agent_data.get("model", "unknown"),
                                            "response": agent_data.get("response", ""),
                                        }
                                    )
                        else:
                            logger.warning(f"Unexpected JSON structure in {debate_file}")

                    except Exception as e:
                        logger.warning(f"Error processing file {debate_file}: {e}")

            except ValueError:
                logger.warning(f"Error processing task directory: {task_dir.name}")

    if not all_data:
        logger.error("Could not find any debate data in directories.")
        return None

    # Create DataFrame from collected data
    df = pd.DataFrame(all_data)
    logger.info(f"Constructed debate data from directories: {len(df)} rows")

    # Write the consolidated data to CSV for future use if requested
    if write_csv and len(df) > 0:
        try:
            df.to_csv(debates_path, index=False)
            logger.info(f"Successfully wrote debate data to {debates_path}")
        except Exception as e:
            logger.warning(f"Failed to write debate_rounds.csv: {e}")

    return df

# -------------------------------------------------------
# Process Distribution Data
# -------------------------------------------------------
def process_distribution_data(
    result_df: pd.DataFrame,
    round_number: int,
) -> Dict[str, float]:
    """Process distribution data to get percentages for each bin (e.g. '0', '1', '2', ...).
    
    Args:
        result_df: DataFrame with distribution data from calculate_correct_rate_distribution_for_round_n.
        round_number: The round number being processed.
        
    Returns:
        Dictionary mapping bin labels (strings) to percentage of tasks in that bin.
    """
    bin_columns = [col for col in result_df.columns if col.isdigit()]
    bin_columns.sort(key=int)
    
    if not bin_columns or result_df.empty:
        logger.warning(f"No bins found for round {round_number}")
        return {}
    
    task_count = len(result_df)
    bin_sums = result_df[bin_columns].sum()
    bin_percentages = (bin_sums / task_count * 100).to_dict()
    
    return bin_percentages

# -------------------------------------------------------
# Plot All Rounds in Two Rows
# -------------------------------------------------------
def plot_all_rounds_two_rows(
    all_distributions: List[Tuple[int, Dict[str, float]]],
    output_dir: Path,
    show_plot: bool = False,
) -> None:
    """
    Plot the round distributions in two rows of subplots.
    Each subplot corresponds to a single round.

    Args:
        all_distributions: List of (round_number, bin_percentages) tuples.
        output_dir: Path to save the resulting figure.
        show_plot: Whether to display the plot interactively.
    """
    if not all_distributions:
        logger.warning("No distributions to plot.")
        return
    
    all_distributions = sorted(all_distributions, key=lambda x: x[0])
    num_rounds = len(all_distributions)

    # We'll lay out subplots in 2 rows. Figure out how many columns we need:
    num_cols = math.ceil(num_rounds / 2)  # 2 rows, so columns = ceil(#rounds / 2)

    fig, axs = plt.subplots(
        nrows=2,
        ncols=num_cols,
        figsize=(5 * num_cols, 10),
        sharey=True  # share the Y-axis for comparison
    )
    axs = axs.ravel()  # Flatten the 2D array of axes into a 1D list

    for i, (round_number, bin_percentages) in enumerate(all_distributions):
        if i >= len(axs):
            break  # Just in case

        ax = axs[i]
        bins = [int(b) for b in sorted(bin_percentages.keys(), key=int)]
        values = [bin_percentages[str(b)] for b in bins]

        bars = ax.bar(bins, values)
        
        # Add text labels on top of each bar
        for bar in bars:
            height = bar.get_height()
            ax.text(
                bar.get_x() + bar.get_width() / 2.0,
                height + 1,
                f"{height:.1f}%",
                ha="center",
                fontsize=9,
            )
        
        ax.set_title(f"Round {round_number}", fontsize=12)
        ax.set_xlabel("Correct Agents", fontsize=10)
        if i % num_cols == 0:
            # Leftmost in each row -> show Y-axis
            ax.set_ylabel("Tasks (%)", fontsize=10)
        else:
            # For aesthetic, you can also hide y-label if you prefer
            pass
        
        if values:
            ax.set_ylim(0, max(values) * 1.2)
        ax.grid(axis="y", alpha=0.3)
    
    # Turn off any extra subplots if we have them
    for j in range(i + 1, len(axs)):
        axs[j].axis("off")

    fig.suptitle("Distribution of Correct Agents per Round (2-row layout)",
                 fontsize=14)
    plt.tight_layout(rect=[0, 0, 1, 0.95])
    
    output_path = output_dir / "all_rounds_two_rows.png"
    plt.savefig(output_path, dpi=300)
    
    if show_plot:
        plt.show()
    plt.close()
    logger.info(f"Saved two-row subplot figure to {output_path}")

# -------------------------------------------------------
# Main Entry Point
# -------------------------------------------------------
def main(
    data_path: Path,
    model_dir: Path,
    output_dir: Path,
    max_rounds: int = 6,
    show_plots: bool = False,
) -> None:
    """
    Loads data, calculates correct-rate distributions for each round,
    and plots them in two rows of subplots.

    Args:
        data_path: Path to the CSV file with task data (contains correct labels).
        model_dir: Directory containing model output data.
        output_dir: Directory where output plots should be saved.
        max_rounds: Maximum number of rounds to process.
        show_plots: Whether to display plots interactively.
    """
    # Create output directory if it doesn't exist
    output_dir.mkdir(parents=True, exist_ok=True)
    
    try:
        # Load the "ground truth" answer data
        df_answers = pd.read_csv(data_path)
        logger.info(f"Loaded answer data from {data_path}")
        
        # Load the debate data
        df_debates = load_debate_data(model_dir)
        if df_debates is None:
            logger.error("Could not load debate data. Aborting.")
            return
        
        logger.info(f"Processed debate data: {len(df_debates)} records")

    except Exception as e:
        logger.error(f"Error loading data: {e}")
        return
    
    all_distributions = []
    
    # Process each round from 0 up to max_rounds-1
    for round_number in range(max_rounds):
        logger.info(f"Processing round {round_number}...")
        
        try:
            # Calculate distribution for this round
            result_df = calculate_correct_rate_distribution_for_round_n(
                df_answers=df_answers,
                df_debates=df_debates,
                round_number=round_number,
                extract_func=extract_caption_a_b_answer,
                compare_func=compare_judge_bench_responses,
            )
            
            # Convert that distribution to a simple dict for plotting
            bin_percentages = process_distribution_data(result_df, round_number)
            
            if bin_percentages:
                all_distributions.append((round_number, bin_percentages))
                
        except Exception as err:
            logger.error(f"Error processing round {round_number}: {err}")
    
    # Create the single, combined plot in 2 rows if we have data
    if all_distributions:
        plot_all_rounds_two_rows(all_distributions, output_dir, show_plot=show_plots)
    
    logger.info("Visualization complete!")

# -------------------------------------------------------
# If run directly
# -------------------------------------------------------
if __name__ == "__main__":
    # Example usage:
    #   1) Load JudgeBench dataset
    #   2) Save to a CSV
    #   3) Run main() with model outputs
    df = load_judge_bench_dataset(dataset_path="../datasets/JudgeBench")
    os.makedirs("../output/judge_bench", exist_ok=True)
    df_path = Path("../output/judge_bench/processed_data.csv")
    df.to_csv(df_path, index=False)
    
    model_dir = Path("../data/judge_bench/llama3(11)")
    OUTPUT_DIR = Path("../output/visualizations/judge_bench")
    
    main(
        data_path=df_path,
        model_dir=model_dir,
        output_dir=OUTPUT_DIR,
        max_rounds=10,
        show_plots=True,  # Set to True if you want to see the plot window
    )
    model_dir = Path("../data/judge_bench/gemma2:2b(11)")
    main(
        data_path=df_path,
        model_dir=model_dir,
        output_dir=OUTPUT_DIR,
        max_rounds=10,
        show_plots=True,  # Set to True if you want to see the plot window
    )


2025-03-26 21:13:53,470 - datasets - INFO - PyTorch version 2.6.0 available.
2025-03-26 21:13:54,493 - __main__ - INFO - Loaded answer data from ../output/judge_bench/processed_data.csv
2025-03-26 21:13:54,498 - multi_llm_debate.analysis.utils - INFO - No debate_rounds.csv found. Attempting to load from directories.
2025-03-26 21:13:54,502 - multi_llm_debate.analysis.utils - INFO - Processing file: debate_round_0.json (Round 0)
2025-03-26 21:13:54,505 - multi_llm_debate.analysis.utils - INFO - Processing file: debate_round_1.json (Round 1)
2025-03-26 21:13:54,509 - multi_llm_debate.analysis.utils - INFO - Processing file: debate_round_2.json (Round 2)
2025-03-26 21:13:54,513 - multi_llm_debate.analysis.utils - INFO - Processing file: debate_round_0.json (Round 0)
2025-03-26 21:13:54,517 - multi_llm_debate.analysis.utils - INFO - Processing file: debate_round_1.json (Round 1)
2025-03-26 21:13:54,520 - multi_llm_debate.analysis.utils - INFO - Processing file: debate_round_2.json (Round 2

Loaded GPT and Claude splits from local paths.


2025-03-26 21:13:54,529 - multi_llm_debate.analysis.utils - INFO - Processing file: debate_round_4.json (Round 4)
2025-03-26 21:13:54,532 - multi_llm_debate.analysis.utils - INFO - Processing file: debate_round_0.json (Round 0)
2025-03-26 21:13:54,536 - multi_llm_debate.analysis.utils - INFO - Processing file: debate_round_0.json (Round 0)
2025-03-26 21:13:54,540 - multi_llm_debate.analysis.utils - INFO - Processing file: debate_round_1.json (Round 1)
2025-03-26 21:13:54,543 - multi_llm_debate.analysis.utils - INFO - Processing file: debate_round_0.json (Round 0)
2025-03-26 21:13:54,547 - multi_llm_debate.analysis.utils - INFO - Processing file: debate_round_6.json (Round 6)
2025-03-26 21:13:54,550 - multi_llm_debate.analysis.utils - INFO - Processing file: debate_round_1.json (Round 1)
2025-03-26 21:13:54,553 - multi_llm_debate.analysis.utils - INFO - Processing file: debate_round_2.json (Round 2)
2025-03-26 21:13:54,556 - multi_llm_debate.analysis.utils - INFO - Processing file: deba